# E015 — Image new augmentation ablation

E007 +All (flip + brightness[0.7,1.3] + noise σ=15) achieved **EER 0.97 ± 0.86%** —
near-perfect on the current data. This experiment tests four additional augmentations
on top of that baseline, motivated by the expected eval distribution shift
(Burget's "schválně zprasené" evaluation set):

1. **+JPEG** (quality 20–50): lossy compression artifacts in degraded captures
2. **+Blur** (σ∈[1.0,2.0]): defocus / motion blur / downsampling
3. **+Rotate** (±10°): head-pose variation across sessions
4. **+Contrast/Gamma** (γ∈[0.5,2.0]): camera exposure / gamma curve differences
5. **+AllNew**: all four combined on top of E007 +All

Methodological guards:
- Augment **only** train fold, never val
- PCA re-fit on augmented train each fold
- E007 +All baseline re-run fresh for consistency

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from scipy.ndimage import gaussian_filter, rotate as ndimage_rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
CONFIG_COLORS = {
    "+All (E007)": "#E74C3C",
    "+All+JPEG":   "#2E86AB",
    "+All+Blur":   "#E67E22",
    "+All+Rotate": "#8E44AD",
    "+All+Contrast": "#27AE60",
    "+All+AllNew": "#1A1A2E",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
N_PCA = 50
C_LOGREG = 1.0
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Augmentation functions

E007 +All baseline functions plus four new ones.
All operate on flat pixel vectors `(6400,)` in [0, 255].

In [ ]:
def find_png(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists():
            return p
    raise FileNotFoundError(stem)

def load_image(png_path: Path) -> np.ndarray:
    """Load PNG → grayscale → flatten. Returns (6400,) in [0, 255]."""
    img = np.array(PILImage.open(png_path).convert("RGB"), dtype=np.float32)
    return img.mean(axis=2).flatten()

def load_images(df: pd.DataFrame, data_dir: Path) -> np.ndarray:
    return np.stack([load_image(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])


# --- E007 +All augmentations ---

def aug_flip(x: np.ndarray) -> np.ndarray:
    return x.reshape(80, 80)[:, ::-1].flatten()

def aug_brightness(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)

def aug_noise_img(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return np.clip(x + rng.normal(0, 15, x.shape), 0, 255)


# --- New augmentations (E015) ---

def aug_jpeg(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """JPEG compression, quality 20-50."""
    quality = int(rng.uniform(20, 50))
    img = PILImage.fromarray(x.reshape(80, 80).astype(np.uint8), mode="L")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return np.array(PILImage.open(buf).convert("L"), dtype=np.float32).flatten()

def aug_blur(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Gaussian blur, sigma in [1.0, 2.0]."""
    sigma = rng.uniform(1.0, 2.0)
    return gaussian_filter(x.reshape(80, 80), sigma=sigma).flatten()

def aug_rotate_img(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Random rotation ±10 degrees."""
    angle = rng.uniform(-10, 10)
    rotated = ndimage_rotate(x.reshape(80, 80), angle, reshape=False, mode="reflect")
    return np.clip(rotated, 0, 255).flatten()

def aug_contrast(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Gamma contrast adjustment, gamma in [0.5, 2.0]."""
    gamma = rng.uniform(0.5, 2.0)
    return np.clip(((x / 255.0) ** gamma) * 255.0, 0, 255)


def augment_dataset(X: np.ndarray, y: np.ndarray, config: str, seed: int) -> tuple:
    """
    Apply augmentation to training data.
    Returns (X_aug, y_aug) = original + augmented copies.
    NEVER call this on val data.

    config options:
      'all'           — E007 +All baseline: flip + brightness + noise (3 extra copies)
      'all_jpeg'      — E007 +All + JPEG (4 extra copies)
      'all_blur'      — E007 +All + Blur (4 extra copies)
      'all_rotate'    — E007 +All + Rotate (4 extra copies)
      'all_contrast'  — E007 +All + Contrast (4 extra copies)
      'all_allnew'    — E007 +All + all four new augs (7 extra copies)
    """
    rng = np.random.default_rng(seed)
    aug_X, aug_y = [], []

    for xi, yi in zip(X, y):
        # E007 +All base: flip, brightness, noise
        base_new = [
            aug_flip(xi),
            aug_brightness(xi, rng),
            aug_noise_img(xi, rng),
        ]

        if config == "all":
            extras = base_new
        elif config == "all_jpeg":
            extras = base_new + [aug_jpeg(xi, rng)]
        elif config == "all_blur":
            extras = base_new + [aug_blur(xi, rng)]
        elif config == "all_rotate":
            extras = base_new + [aug_rotate_img(xi, rng)]
        elif config == "all_contrast":
            extras = base_new + [aug_contrast(xi, rng)]
        elif config == "all_allnew":
            extras = base_new + [
                aug_jpeg(xi, rng),
                aug_blur(xi, rng),
                aug_rotate_img(xi, rng),
                aug_contrast(xi, rng),
            ]
        else:
            raise ValueError(f"Unknown config: {config}")

        for xa in extras:
            aug_X.append(xa)
            aug_y.append(yi)

    X_out = np.vstack([X, np.stack(aug_X)])
    y_out = np.concatenate([y, np.array(aug_y)])
    return X_out, y_out

print("Augmentation functions defined.")

## 2. Augmentation visualization

Visual sanity check — do the new augmentations produce realistic face images?

In [ ]:
rng_viz = np.random.default_rng(42)
sample_row = manifest[manifest.label == 1].iloc[0]
orig = load_image(find_png(sample_row["stem"], DATA))

augmented = {
    "Original":         orig,
    "Flip":             aug_flip(orig),
    "Brightness\n[0.7,1.3]": aug_brightness(orig, np.random.default_rng(42)),
    "Noise\n(σ=15)":    aug_noise_img(orig, np.random.default_rng(42)),
    "JPEG\n(q=20-50)":  aug_jpeg(orig, np.random.default_rng(42)),
    "Blur\n(σ∈[1,2])": aug_blur(orig, np.random.default_rng(42)),
    "Rotate\n(±10°)":   aug_rotate_img(orig, np.random.default_rng(42)),
    "Contrast\n(γ∈[0.5,2])": aug_contrast(orig, np.random.default_rng(42)),
}

fig, axes = plt.subplots(1, 8, figsize=(20, 3.2))
for ax, (title, img) in zip(axes, augmented.items()):
    ax.imshow(img.reshape(80, 80), cmap="gray", vmin=0, vmax=255)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

plt.suptitle(f"E015 — All augmentations on {sample_row['stem']} (target)",
             color=COLORS["target"], fontsize=10)
plt.tight_layout()
plt.show()

## 3. Cross-validation across all configs

Each config runs full LOSO CV (3 folds, seed=67).
Augmentation applied only inside the fold loop, after train/val split.
Val always uses original images.

In [ ]:
CONFIGS = {
    "all":          "+All (E007)",
    "all_jpeg":     "+All+JPEG",
    "all_blur":     "+All+Blur",
    "all_rotate":   "+All+Rotate",
    "all_contrast": "+All+Contrast",
    "all_allnew":   "+All+AllNew",
}

all_results = {}  # config_name → list of fold dicts
all_oof     = {}  # config_name → oof_scores array

for config_key, config_name in CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Config: {config_name}")
    print('='*55)

    oof_scores   = np.full(len(manifest), np.nan)
    fold_results = []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]
        y_train  = train_df["label"].to_numpy()
        y_val    = val_df["label"].to_numpy()

        # Load original images
        X_train_orig = load_images(train_df, DATA)
        X_val        = load_images(val_df,   DATA)  # val: originals only

        # Augment train fold only
        X_train, y_train_aug = augment_dataset(X_train_orig, y_train, config_key, seed=SEED + fold_id)

        # Scaler + PCA fitted on augmented train
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_val_s   = scaler.transform(X_val)

        pca = PCA(n_components=N_PCA, random_state=SEED)
        X_train_pca = pca.fit_transform(X_train_s)
        X_val_pca   = pca.transform(X_val_s)

        clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
        clf.fit(X_train_pca, y_train_aug)

        val_scores = clf.decision_function(X_val_pca)
        oof_scores[val_idx] = val_scores

        eer, _     = compute_eer(val_scores[y_val == 1], val_scores[y_val == 0])
        min_dcf, _ = compute_min_dcf(val_scores[y_val == 1], val_scores[y_val == 0])
        fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf,
                              "n_train": len(X_train)})
        print(f"  Fold {fold_id}: train={len(X_train)} (aug), val={len(X_val)} → "
              f"EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

    all_results[config_name] = fold_results
    all_oof[config_name]     = oof_scores.copy()

print("\nAll configs done.")

## 4. Results — ablation table

In [ ]:
print(f"{'Config':<18} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 74)

summary = []
baseline_mean = None

for config_name, fold_results in all_results.items():
    eers   = [r["eer"] * 100 for r in fold_results]
    dcfs   = [r["min_dcf"]   for r in fold_results]
    mean_e = np.mean(eers)
    std_e  = np.std(eers)
    mean_d = np.mean(dcfs)

    marker = " ★" if config_name == "+All (E007)" else ""
    print(f"{config_name + marker:<18} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
          f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}")
    summary.append({"config": config_name, "f0": eers[0], "f1": eers[1], "f2": eers[2],
                    "mean": mean_e, "std": std_e, "min_dcf": mean_d})
    if config_name == "+All (E007)":
        baseline_mean = mean_e

print("-" * 74)

best = min(summary, key=lambda x: x["mean"])
print(f"Best config: {best['config']}  EER={best['mean']:.2f}±{best['std']:.2f}%")

if best["config"] != "+All (E007)":
    delta = best["mean"] - baseline_mean
    print(f"Delta vs E007 +All baseline: {delta:+.2f}% EER")
else:
    print("E007 +All baseline wins — no new aug improves it")

# OOF overall for best config
oof_best = all_oof[best["config"]]
eer_oof, _   = compute_eer(oof_best[y_all == 1], oof_best[y_all == 0])
dcf_oof, thr = compute_min_dcf(oof_best[y_all == 1], oof_best[y_all == 0])
print(f"\nBest OOF overall: EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 5. Visualizations

In [ ]:
# Mean ± std bar chart
fig, ax = plt.subplots(figsize=(10, 5))

configs = list(all_results.keys())
means  = [np.mean([r["eer"] * 100 for r in all_results[c]]) for c in configs]
stds   = [np.std( [r["eer"] * 100 for r in all_results[c]]) for c in configs]
colors = [CONFIG_COLORS[c] for c in configs]

bars = ax.bar(range(len(configs)), means, color=colors, alpha=0.85,
              yerr=stds, capsize=7, error_kw=dict(elinewidth=2))

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2, m + s + 0.05,
            f"{m:.2f}\n±{s:.2f}", ha="center", fontsize=8)

best_idx = np.argmin(means)
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("★ best",
            xy=(best_idx, means[best_idx] - stds[best_idx] - 0.15),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")

# Reference line for E007 +All
ax.axhline(means[0], color=CONFIG_COLORS["+All (E007)"], ls="--", lw=1.5,
           label=f"E007 +All = {means[0]:.2f}%")

ax.set_xticks(range(len(configs)))
ax.set_xticklabels(configs, fontsize=9, rotation=15, ha="right")
ax.set_ylabel("Mean EER [%] ± std")
ax.set_title("E015 — Mean EER per augmentation config (LOSO, 3 folds)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Per-fold grouped bar chart
fig, ax = plt.subplots(figsize=(12, 5))

n_folds   = 3
n_configs = len(configs)
x         = np.arange(n_folds)
width     = 0.13

for i, (config_name, fold_results) in enumerate(all_results.items()):
    eers   = [r["eer"] * 100 for r in fold_results]
    offset = (i - n_configs / 2 + 0.5) * width
    ax.bar(x + offset, eers, width,
           label=config_name,
           color=CONFIG_COLORS[config_name],
           alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(["Fold 0 (session 01)", "Fold 1 (session 02)", "Fold 2 (session 03)"])
ax.set_ylabel("EER [%]")
ax.set_title("E015 — Per-fold EER by augmentation config")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# DET curves overlay (OOF)
fig, ax = plt.subplots(figsize=(8, 7))

ticks     = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos  = [scipy_norm.ppf(t) for t in ticks]
tick_labs = [f"{int(t * 100)}" for t in ticks]

best_name = best["config"]

for config_name, oof_s in all_oof.items():
    valid = ~np.isnan(oof_s)
    fpr, tpr, _ = roc_curve(y_all[valid], oof_s[valid])
    far_c = np.clip(fpr,   1e-4, 1 - 1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1 - 1e-4)
    eer_c, _ = compute_eer(oof_s[valid & (y_all == 1)], oof_s[valid & (y_all == 0)])
    lw = 2.5 if config_name == best_name else 1.5
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=CONFIG_COLORS[config_name], lw=lw,
            label=f"{config_name}  EER={eer_c*100:.1f}%",
            zorder=5 if config_name == best_name else 1)

ax.plot(tick_pos, tick_pos, "k--", lw=1, label="EER line")
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labs)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labs)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("E015 — DET curves, all configs (OOF)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary table as DataFrame for easy reading
df_summary = pd.DataFrame(summary)
df_summary["mean±std"] = df_summary.apply(
    lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1
)
df_summary["vs_baseline"] = df_summary["mean"] - df_summary.loc[
    df_summary["config"] == "+All (E007)", "mean"
].values[0]
df_summary["vs_baseline"] = df_summary["vs_baseline"].apply(
    lambda x: f"{x:+.2f}%"
)

display_cols = ["config", "f0", "f1", "f2", "mean±std", "min_dcf", "vs_baseline"]
print(df_summary[display_cols].to_string(index=False))

print(f"\nE007 reference row: 2.08 / 0.83 / 0.00 / 0.97 ± 0.86 / 0.0194")
print(f"Winner: {best['config']}  EER={best['mean']:.2f} ± {best['std']:.2f}%  min-DCF={best['min_dcf']:.4f}")